In [5]:
import pandas as pd

po_header = pd.read_csv('../data/raw/purchase_orders_header.csv')
po_header.head()


,po_id,supplier_id,branch_id,order_date,expected_delivery_date,received_date,po_status,total_cost,total_gst_amount,grand_total
0,PO-141186,SUP0001,PUN001,2019-04-10,2019-04-29,2019-04-29,Received,22955232.11,6.427465e+06,2.938270e+07
1,PO-215234,SUP0005,DEL001,2020-12-30,2021-01-27,2021-01-27,Received,19406544.39,5.428358e+06,2.483490e+07
2,PO-296826,SUP0008,AHM001,2022-04-05,2022-05-01,2022-05-01,Received,1295425.79,3.627192e+05,1.658145e+06
3,PO-700652,SUP0004,AHM001,2019-07-29,2019-08-16,2019-08-17,Received,38753295.34,1.080312e+07,4.955641e+07
4,PO-120358,SUP0004,DEL001,2024-05-14,2024-06-01,2024-06-01,Received,614701.45,1.106463e+05,7.253477e+05


In [6]:
print("=== INFO ===")
print(po_header.info())

print("\n=== MISSING VALUES ===")
print(po_header.isnull().sum())

print("\n=== SAMPLE DATA ===")
po_header.head()


=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   po_id                   24000 non-null  str    
 1   supplier_id             24000 non-null  str    
 2   branch_id               24000 non-null  str    
 3   order_date              24000 non-null  str    
 4   expected_delivery_date  24000 non-null  str    
 5   received_date           21630 non-null  str    
 6   po_status               24000 non-null  str    
 7   total_cost              24000 non-null  float64
 8   total_gst_amount        24000 non-null  float64
 9   grand_total             24000 non-null  float64
dtypes: float64(3), str(7)
memory usage: 1.8 MB
None

=== MISSING VALUES ===
po_id                        0
supplier_id                  0
branch_id                    0
order_date                   0
expected_delivery_date       0
received_date        

,po_id,supplier_id,branch_id,order_date,expected_delivery_date,received_date,po_status,total_cost,total_gst_amount,grand_total
0,PO-141186,SUP0001,PUN001,2019-04-10,2019-04-29,2019-04-29,Received,22955232.11,6.427465e+06,2.938270e+07
1,PO-215234,SUP0005,DEL001,2020-12-30,2021-01-27,2021-01-27,Received,19406544.39,5.428358e+06,2.483490e+07
2,PO-296826,SUP0008,AHM001,2022-04-05,2022-05-01,2022-05-01,Received,1295425.79,3.627192e+05,1.658145e+06
3,PO-700652,SUP0004,AHM001,2019-07-29,2019-08-16,2019-08-17,Received,38753295.34,1.080312e+07,4.955641e+07
4,PO-120358,SUP0004,DEL001,2024-05-14,2024-06-01,2024-06-01,Received,614701.45,1.106463e+05,7.253477e+05


In [7]:
text_cols = [
    'po_id', 'supplier_id', 'branch_id', 'po_status'
]

for col in text_cols:
    po_header[col] = po_header[col].str.upper()


In [8]:
po_header['order_date'] = pd.to_datetime(po_header['order_date'], errors='coerce')
po_header['expected_delivery_date'] = pd.to_datetime(po_header['expected_delivery_date'], errors='coerce')
po_header['received_date'] = pd.to_datetime(po_header['received_date'], errors='coerce')

po_header[['order_date', 'expected_delivery_date', 'received_date']].head()


,order_date,expected_delivery_date,received_date
0,2019-04-10,2019-04-29,2019-04-29
1,2020-12-30,2021-01-27,2021-01-27
2,2022-04-05,2022-05-01,2022-05-01
3,2019-07-29,2019-08-16,2019-08-17
4,2024-05-14,2024-06-01,2024-06-01


In [9]:
po_header['received_date'] = po_header['received_date'].fillna(pd.NaT)


In [14]:
po_header['lead_time_days'] = (
    (po_header['received_date'] - po_header['order_date']).dt.days
)


po_header['delay_days'] = (
    (po_header['received_date'] - po_header['expected_delivery_date']).dt.days
)

po_header['delay_flag'] = (
    (po_header['delay_days'] > 0)
).astype(int)


po_header['on_time_flag'] = (
    (po_header['delay_days'] <= 0)
).astype(int)


po_header['gst_pct'] = (
    po_header['total_gst_amount'] / po_header['total_cost']
)



In [15]:
po_header.drop_duplicates(inplace=True)


In [16]:
print("=== DUPLICATE PO IDs ===")
print(po_header['po_id'].duplicated().sum())

print("\n=== PO STATUS VALUES ===")
print(po_header['po_status'].unique())

print("\n=== DATE CHECK ===")
print(po_header[['order_date', 'expected_delivery_date', 'received_date']].head())

print("\n=== NUMERIC SUMMARY ===")
numeric_cols = ['total_cost', 'total_gst_amount', 'grand_total']
print(po_header[numeric_cols].describe())


=== DUPLICATE PO IDs ===
0

=== PO STATUS VALUES ===
<StringArray>
['RECEIVED', 'CANCELLED']
Length: 2, dtype: str

=== DATE CHECK ===
  order_date expected_delivery_date received_date
0 2019-04-10             2019-04-29    2019-04-29
1 2020-12-30             2021-01-27    2021-01-27
2 2022-04-05             2022-05-01    2022-05-01
3 2019-07-29             2019-08-16    2019-08-17
4 2024-05-14             2024-06-01    2024-06-01

=== NUMERIC SUMMARY ===
         total_cost  total_gst_amount   grand_total
count  2.400000e+04      2.400000e+04  2.400000e+04
mean   1.444258e+07      3.961014e+06  1.840360e+07
std    1.223942e+07      3.409115e+06  1.564760e+07
min    5.675010e+03      1.021502e+03  6.696512e+03
25%    4.775103e+06      1.271323e+06  6.050990e+06
50%    1.153000e+07      3.125566e+06  1.467005e+07
75%    2.104935e+07      5.785362e+06  2.684122e+07
max    8.712161e+07      2.439405e+07  1.115157e+08


In [17]:
po_header.to_csv('../data/processed/purchase_orders_header.csv', index=False)
print("Processed purchase_orders_header.csv saved to data/processed/")


Processed purchase_orders_header.csv saved to data/processed/
